# Chapter 11 — Attention from Scratch

Runnable companion to `docs/10_attention_mechanism.md`. We build
**scaled dot-product attention** with bare PyTorch (no
`nn.MultiheadAttention`), visualize the attention matrix on a toy
sentence, and inspect what multiple heads see.

Generated by `src/build_chapter_10_attention_toy.py`.

## Setup

In [1]:
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cpu")
print("torch", torch.__version__, "device", DEVICE)

torch 2.12.0+cu130 device cpu


## 1. Scaled dot-product attention from scratch

For Q, K of dimension `d_k`:

```
Attention(Q, K, V) = softmax( Q · K^T / sqrt(d_k) ) · V
```

Returning the weights as well as the output makes the function easy to
visualize.

In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q, K: (B, T_q, d_k);  V: (B, T_k, d_v)
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)        # (B, T_q, T_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)                      # (B, T_q, T_k)
    return weights @ V, weights                              # ((B, T_q, d_v), (B, T_q, T_k))


# Smoke test on random tensors.
B, T, d = 2, 5, 8
Q = torch.randn(B, T, d); K = torch.randn(B, T, d); V = torch.randn(B, T, d)
out, w = scaled_dot_product_attention(Q, K, V)
print("Q/K/V shape:", Q.shape, "out:", out.shape, "weights:", w.shape)
print("each row of weights sums to 1:", w.sum(dim=-1)[0])

Q/K/V shape: torch.Size([2, 5, 8]) out: torch.Size([2, 5, 8]) weights: torch.Size([2, 5, 5])
each row of weights sums to 1: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 2. Causal mask

For language modeling, position `t` should not attend to positions
`> t`. The lower-triangular mask zeroes out the future.

In [3]:
T = 6
mask = torch.tril(torch.ones(T, T)).bool()
print("causal mask (1 = visible, 0 = blocked):")
print(mask.int())

causal mask (1 = visible, 0 = blocked):
tensor([[1, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1]], dtype=torch.int32)


## 3. Visualize attention on a toy sentence

We *hand-craft* embeddings for six tokens so the attention pattern is
predictable. Each token is a 4-D one-hot-like vector; we set up Q and K
so that "cat" and "dog" share a feature, and "the" attends to whichever
noun is nearby.

In [4]:
TOKENS = ["the", "cat", "sat", "on", "the", "mat"]

# Hand-crafted 4-D embeddings: column 0 = "noun-likeness", column 1 = "place-likeness",
# column 2 = "determiner", column 3 = "verb-likeness".
embed_table = {
    "the": torch.tensor([0.0, 0.0, 1.0, 0.0]),
    "cat": torch.tensor([1.0, 0.0, 0.0, 0.0]),
    "sat": torch.tensor([0.0, 0.0, 0.0, 1.0]),
    "on":  torch.tensor([0.0, 0.5, 0.0, 0.0]),
    "mat": torch.tensor([0.7, 0.7, 0.0, 0.0]),
}

X = torch.stack([embed_table[t] for t in TOKENS]).unsqueeze(0)    # (1, 6, 4)
print("X shape:", tuple(X.shape))

X shape: (1, 6, 4)


In [5]:
# Identity Q/K/V projections so the meaning is clearer (no learned weights).
Q = X.clone(); K = X.clone(); V = X.clone()

out, weights = scaled_dot_product_attention(Q, K, V)
weights_2d = weights[0].numpy()                                  # (6, 6)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(weights_2d, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(TOKENS))); ax.set_xticklabels(TOKENS, rotation=45)
ax.set_yticks(range(len(TOKENS))); ax.set_yticklabels(TOKENS)
ax.set_xlabel("key (attended to)")
ax.set_ylabel("query (attending)")
ax.set_title("Attention matrix — identity Q/K/V on hand-crafted embeddings")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

# Print the sharpest attention from each row.
for i, q in enumerate(TOKENS):
    j = int(weights_2d[i].argmax())
    print(f"'{q}' attends most to '{TOKENS[j]}' "
          f"(weight={weights_2d[i, j]:.2f})")

'the' attends most to 'the' (weight=0.23)
'cat' attends most to 'cat' (weight=0.23)
'sat' attends most to 'sat' (weight=0.25)
'on' attends most to 'mat' (weight=0.19)
'the' attends most to 'the' (weight=0.23)
'mat' attends most to 'mat' (weight=0.23)


/tmp/ipykernel_355175/3566254421.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


With identity Q/K/V, each token attends most to *itself* (and to other
tokens that share its features). "cat" and "mat" both share the noun
feature, so they pick each other up too. Real Transformers learn the
Q/K/V projections so the attention pattern becomes task-relevant.

## 4. Causal-masked attention on the same input

In [6]:
out, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
weights_2d = weights[0].numpy()

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(weights_2d, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(TOKENS))); ax.set_xticklabels(TOKENS, rotation=45)
ax.set_yticks(range(len(TOKENS))); ax.set_yticklabels(TOKENS)
ax.set_title("Attention matrix with causal mask")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()
print("upper-triangle weights are zero (no future-token leakage).")

upper-triangle weights are zero (no future-token leakage).


/tmp/ipykernel_355175/3370857770.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Multi-head attention

Run `h` heads in parallel, each projecting into `d_k = d_model // h`
dimensions, then concatenate. Even *random* projections give different
heads with different attention patterns — that is the whole point of
multi-head: cheaper diversity.

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h  = num_heads
        self.dk = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def split(self, t):
        B, T, D = t.shape
        return t.view(B, T, self.h, self.dk).transpose(1, 2)        # (B, h, T, dk)

    def forward(self, x, mask=None):
        Q = self.split(self.W_Q(x))
        K = self.split(self.W_K(x))
        V = self.split(self.W_V(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        w = F.softmax(scores, dim=-1)             # (B, h, T, T)
        out = (w @ V).transpose(1, 2).contiguous().view(x.shape)
        return self.W_O(out), w


torch.manual_seed(0)
mha = MultiHeadAttention(d_model=4, num_heads=2)
out, w = mha(X)
print("output shape:", tuple(out.shape), "weights shape:", tuple(w.shape))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for h_idx, ax in enumerate(axes):
    im = ax.imshow(w[0, h_idx].detach().numpy(), cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(TOKENS))); ax.set_xticklabels(TOKENS, rotation=45)
    ax.set_yticks(range(len(TOKENS))); ax.set_yticklabels(TOKENS)
    ax.set_title(f"head {h_idx}")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

output shape: (1, 6, 4) weights shape: (1, 2, 6, 6)


/tmp/ipykernel_355175/2981093940.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The two heads attend to *different* positions on the same input even
without any training — different random projections produce different
similarity patterns. After training on a real task, this diversity
becomes task-specific (one head tracks subject-verb agreement, another
tracks coreference, and so on).

## Quick check

- Why divide by `sqrt(d_k)`?
  As `d_k` grows, the variance of `Q · K` grows linearly with `d_k`.
  Without rescaling, the softmax saturates (one cell ≈ 1, the rest ≈ 0)
  and the gradient through softmax collapses.
- Why a separate `W_O`?
  After concatenating heads we have `d_model` features back, but in an
  arbitrary order. `W_O` learns the right linear mix for the next sub-layer.